## Classification Models

In [16]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing  import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

df=pd.read_csv("../data/student_dataset/student-mat.csv", sep=";")
df.head(5)

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [17]:
#Feature selection and training/testing
df["passed"]=(df["G3"]>=10).astype(int)
X=df.drop(columns=["G3", "passed"])
y=df["passed"]
X_train, X_test, y_train, y_test= train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
numerical_cols=X.select_dtypes(include="number").columns
categorical_cols=X.select_dtypes(include=["object", "category"]).columns
numerical_transformer=StandardScaler()
categorical_cols_transformer=OneHotEncoder()
preprocessor=ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_cols),
        ("cat", categorical_cols_transformer, categorical_cols)
    ]
)

C:\Users\anamo\AppData\Local\Temp\ipykernel_23000\176136636.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols=X.select_dtypes(include=["object", "category"]).columns


In [18]:
#Model Training
models= {
    "logistic":Pipeline([
        ("preprocessor", preprocessor),
        ("logistic", LogisticRegression(max_iter=10000))
    ]),
    "knn":Pipeline([
    ("preprocessor", preprocessor),
    ("knn", KNeighborsClassifier(n_neighbors=10))
    ]),
    "decision":Pipeline([
        ("preprocessor", preprocessor),
        ("decision",DecisionTreeClassifier(
                criterion="gini",
                max_depth=5,
                random_state=42
            ))
    ]),
    "random":Pipeline([
        ("preprocessor", preprocessor),
       ("random",  RandomForestClassifier(
               n_estimators=100,
               max_depth=5,
               random_state=42
           ))
    ])
}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred=model.predict(X_test)
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))
    print(classification_report(
        y_test, y_pred,
        labels=[0, 1],
        target_names=["did not pass", "passed"],
        zero_division=0
    ))
    print(f"Accuracy for {name}: {accuracy_score(y_test, y_pred):.2f}")
    print(f"F1 for did not pass: {f1_score(y_test, y_pred, pos_label=0):.2f}")
    print(f"Precision for did not pass: {precision_score(y_test, y_pred, pos_label=0):.2f}")
    print(f"Recall for did not pass: {recall_score(y_test, y_pred, pos_label=0):.2f}")
    

[[24  2]
 [ 8 45]]
              precision    recall  f1-score   support

did not pass       0.75      0.92      0.83        26
      passed       0.96      0.85      0.90        53

    accuracy                           0.87        79
   macro avg       0.85      0.89      0.86        79
weighted avg       0.89      0.87      0.88        79

Accuracy for logistic: 0.87
F1 for did not pass: 0.83
Precision for did not pass: 0.75
Recall for did not pass: 0.92
[[17  9]
 [ 6 47]]
              precision    recall  f1-score   support

did not pass       0.74      0.65      0.69        26
      passed       0.84      0.89      0.86        53

    accuracy                           0.81        79
   macro avg       0.79      0.77      0.78        79
weighted avg       0.81      0.81      0.81        79

Accuracy for knn: 0.81
F1 for did not pass: 0.69
Precision for did not pass: 0.74
Recall for did not pass: 0.65
[[25  1]
 [ 8 45]]
              precision    recall  f1-score   support

did n

## Results and interpretation

On this train/test split, the Decision Tree had the highest recall for students who did not pass (0.96): it identified 25 of 26 such students and missed 1. It also incorrectly flagged 8 students who passed. Logistic Regression identified 24 of 26 students who did not pass, while KNN identified 17 of 26. For this project, recall for the “did not pass” class matters because missed at-risk students are the main concern. These results come from one split; Stage 5 will compare models with cross-validation and tuning.

This notebook keeps G1 and G2 as inputs, so it represents a prediction made after those grades are available.
